# 🏆 [Day 33] 실전 GDS 복합 투영 & 신약 타겟팅 다차원 중심성 핸즈온 워크북

> **미션 개요**:
> Hetionet 바이오 메디컬 지식그래프(15,540개 노드, 91,966건 엣지)를 활용하여 **실무 엔터프라이즈급 GDS 인메모리 삼각 투영, 3대 중심성 교차 비교, 그리고 타겟 질환 관점의 개인화 PageRank(PPR)**를 완벽히 실습합니다.
>
> 1. **다중 이종 노드·관계 삼각 투영**: `Disease` + `Gene` + `Compound` 노드와 `ASSOCIATES` + `BINDS` + `TREATS` 엣지 결합 투영
> 2. **3대 중심성 지표 교차 비교**: Degree(마당발) vs PageRank(실세) vs Betweenness(길목) 비교 분석
> 3. **타겟 질환 중심의 개인화 PageRank (PPR)**: 유방암(`breast cancer`) 및 당뇨병(`type 2 diabetes mellitus`) 타겟 최적 약물 후보군 랭킹 도출
> 4. **GDS 수명주기 관리**: 안전한 인메모리 메모리 반환(`drop`) 보장

## 0. 환경 설정 및 Neo4j 연결

In [ ]:
import os
import pandas as pd
from dotenv import load_dotenv
from neo4j import GraphDatabase

load_dotenv(".env")
load_dotenv("../.env")

NEO4J_URI = os.getenv("NEO4J_URI", "bolt://localhost:7687")
NEO4J_USER = os.getenv("NEO4J_USER", "neo4j")
NEO4J_PASSWORD = os.getenv("NEO4J_PASSWORD")

driver = GraphDatabase.driver(NEO4J_URI, auth=(NEO4J_USER, NEO4J_PASSWORD))

def run_cypher(query, **params):
    with driver.session() as session:
        return [record.data() for record in session.run(query, **params)]

ver_res = run_cypher("RETURN gds.version() AS ver")[0]
print(f"✅ Neo4j GDS 플러그인 버전: {ver_res['ver']}")

## 1. 다중 이종 노드·관계 삼각 투영 (gds.graph.project)
- 노드: `Disease`, `Gene`, `Compound` 3개 레이블
- 관계: `ASSOCIATES`, `BINDS`, `TREATS` 3개 관계를 모두 `UNDIRECTED`로 투영

In [ ]:
# 기존 투영이 있으면 안전하게 해제
run_cypher("CALL gds.graph.drop('bioMasterGraph', false) YIELD graphName")

proj_cypher = """
CALL gds.graph.project(
    'bioMasterGraph',
    ['Disease', 'Gene', 'Compound'],
    {
        ASSOCIATES: {type: 'ASSOCIATES', orientation: 'UNDIRECTED'},
        BINDS: {type: 'BINDS', orientation: 'UNDIRECTED'},
        TREATS: {type: 'TREATS', orientation: 'UNDIRECTED'}
    }
)
YIELD graphName, nodeCount, relationshipCount, projectMillis
"""
res = run_cypher(proj_cypher)[0]
print(f"✅ 인메모리 삼각 투영 완료: {res['graphName']}")
print(f"  • 투영된 노드 수: {res['nodeCount']:,}개")
print(f"  • 투영된 엣지 수: {res['relationshipCount']:,}건 (무방향 2배 검산 적용)")
print(f"  • 소요 시간: {res['projectMillis']} ms")

## 2. 3대 중심성 지표(Degree vs PageRank vs Betweenness) 교차 비교 분석

In [ ]:
# 1. 차수 중심성 (Degree: 마당발)
deg_df = pd.DataFrame(run_cypher("""
CALL gds.degree.stream('bioMasterGraph')
YIELD nodeId, score
WITH gds.util.asNode(nodeId) AS n, score
RETURN n.name AS name, labels(n)[0] AS type, toInteger(score) AS degree
ORDER BY degree DESC LIMIT 5
"""))

# 2. PageRank 중심성 (실세/권력자)
pr_df = pd.DataFrame(run_cypher("""
CALL gds.pageRank.stream('bioMasterGraph', {maxIterations: 20, dampingFactor: 0.85})
YIELD nodeId, score
WITH gds.util.asNode(nodeId) AS n, score
RETURN n.name AS name, labels(n)[0] AS type, round(score, 4) AS pagerank
ORDER BY pagerank DESC LIMIT 5
"""))

# 3. 매개 중심성 (Betweenness: 길목/브로커)
btw_df = pd.DataFrame(run_cypher("""
CALL gds.betweenness.stream('bioMasterGraph')
YIELD nodeId, score
WITH gds.util.asNode(nodeId) AS n, score
RETURN n.name AS name, labels(n)[0] AS type, round(score, 2) AS betweenness
ORDER BY betweenness DESC LIMIT 5
"""))

print("📊 [1. 차수 중심성 (Degree) Top 5 - 단순 연결 수가 많은 노드]")
display(deg_df)

print("\n🌐 [2. PageRank Top 5 - 전역 영향력이 높은 핵심 허브]")
display(pr_df)

print("\n🌉 [3. 매개 중심성 (Betweenness) Top 5 - 네트워크의 핵심 길목]")
display(btw_df)

## 3. 타겟 질환 관점의 개인화 PageRank (PPR) 신약 후보군 도출
- 유방암(`breast cancer`)을 출발점으로 지정하여, 가장 강하게 연결되는 화합물(약물) 후보군 10개를 선별합니다.

In [ ]:
target_disease = 'breast cancer'

ppr_res = run_cypher("""
MATCH (d:Disease {name: $disease_name})
WITH collect(id(d)) AS sources
CALL gds.pageRank.stream('bioMasterGraph', {
    maxIterations: 20,
    dampingFactor: 0.85,
    sourceNodes: sources
})
YIELD nodeId, score
WITH gds.util.asNode(nodeId) AS n, score
WHERE n:Compound
RETURN n.name AS compound_name, round(score, 6) AS ppr_score
ORDER BY ppr_score DESC
LIMIT 10
""", disease_name=target_disease)

df_ppr = pd.DataFrame(ppr_res)
print(f"🎯 [{target_disease} 타겟 개인화 PageRank Top 10 약물 후보]")
display(df_ppr)

## 4. 인메모리 투영 메모리 안전 해제 (Drop)

In [ ]:
drop_res = run_cypher("CALL gds.graph.drop('bioMasterGraph') YIELD graphName")[0]
print(f"✅ 인메모리 투영 메모리 정상 반환 완료: {drop_res['graphName']}")